[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/18_embedding_interview.ipynb)

# 🟢 Solution: Embedding Layer（面试版）

## 📋 题目描述

**难度：** Easy

实现嵌入查找层（nn.Module）。

嵌入层通过可学习的权重矩阵将整数索引映射为稠密向量，是 NLP 模型的第一层。

**签名:** `MyEmbedding(num_embeddings, embedding_dim)`（nn.Module）

**前向传播:** `forward(indices) -> Tensor`
- `indices` — 任意形状的整数张量

**返回:** 嵌入向量，末尾增加 embedding_dim 维度

**约束:**
- 权重存储为 `nn.Parameter`，形状 (num_embeddings, embedding_dim)
- 前向传播即 `weight[indices]`

**提示：** `self.weight = nn.Parameter(...)` 形状 `(num_embeddings, embedding_dim)`
前向：`return self.weight[indices]`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        # ---- Step 1: 嵌入矩阵 ----
        # 本质就是一个查找表：token_id → dense_vector
        # num_embeddings: 词表大小 V
        # embedding_dim: 向量维度 D
        # 总参数量 = V × D（比如 GPT-2: 50257 × 768 ≈ 38.6M）
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))

    def forward(self, indices):
        # ---- Step 2: 查表 ----
        # 面试要点：Embedding = one-hot × weight 的高效实现
        # 数学上：y = one_hot(indices) @ self.weight
        # 但 one_hot 编码浪费内存（V 维 one-hot × V×D 矩阵 = 只取一行）
        # PyTorch 的高级索引 weight[indices] 直接取对应行，等价但高效
        #
        # 反向传播：只有被查到的行收到梯度，其余行为 0
        # 这是稀疏梯度——PyTorch 内部用 scatter 操作实现
        return self.weight[indices]

# 面试常见追问：
# Q: Embedding 和 Linear 的区别？
# A: Embedding 是查表（索引 → 向量），Linear 是矩阵乘法（向量 → 向量）
#    Embedding 等价于 one_hot(x) @ weight，但实现上跳过了 one_hot
# Q: 为什么不用 one_hot + matmul？
# A: 内存和计算都浪费。one_hot 产生 V 维稀疏向量，99.99% 是 0。
#    直接索引查表 O(1)，one_hot+matmul O(V)。


In [ ]:
emb = MyEmbedding(num_embeddings=100, embedding_dim=16)
idx = torch.tensor([1, 5, 99, 42])
print('Output:', emb(idx).shape)
print('Same row:', torch.equal(emb(torch.tensor([5])), emb(torch.tensor([5]))))


In [ ]:
from torch_judge import check
check('embedding')


## 📝 核心思路总结

1. **本质是查表**：Embedding 层就是一个可学习的查找表 (V, D)
2. **高级索引**：`weight[indices]` 等价于 `one_hot @ weight`，但高效得多
3. **稀疏梯度**：反向传播时只有被查到的行收到梯度
4. **vs Linear**：Embedding 是索引查表，Linear 是矩阵乘法
